In [1]:
# ===== 1: 설정 =====
import json
from pathlib import Path
from collections import defaultdict

BASE_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
GRAPH_DIR = BASE_DIR / "data" / "graph"

def load_resolved(species: str) -> list[dict]:
    with open(GRAPH_DIR / f"petmint_{species}_resolved.json", "r", encoding="utf-8") as f:
        return json.load(f)

chunks_dog = load_resolved("강아지")
chunks_cat = load_resolved("고양이")

print("강아지 청크 수:", len(chunks_dog))
print("고양이 청크 수:", len(chunks_cat))

강아지 청크 수: 928
고양이 청크 수: 850


In [2]:
# ===== 2: 조각(청크) 단위 참조 그래프 (ref_* 필드 사용; inline_law/not_found/law_not_in_excerpt/self_*_not_found/self_item 제외) =====
# 노드 키 = 섹션|문서|조|등급|항|조각 — 그 청크가 '직접' 인용한 조문만 붙는다.
# (조 단위 키는 4항 결과에 7항이 인용한 조문까지 붙였고, 항 단위 키도 분할된 항의 다른 조각이 인용한 조문을 붙였다)
# 엣지 = [조, 제목, 섹션, 문서, 항, 호]. 항 자리에는 paragraph_type=='항'일 때만 번호를 넣는다
# — '제5조제2호'의 2는 호이지 2항이 아니다(호를 항으로 조회해 다른 항이 붙던 문제, 2026-10-12).

GRAPH_SCOPES = {
    "same_section", "same_section_same_type", "same_section_diff_type",
    "cross_section", "self_paragraph", "law_section_reference", "named_document",
}

def build_reference_graph(chunks: list[dict]) -> dict:
    graph = defaultdict(set)
    for c in chunks:
        if c.get("article_num") is None:
            continue
        src = (c["section"], c.get("section_name"), c["article_num"], c.get("coverage_type"), c.get("paragraph_num"),
               c.get("chunk_index"))
        for cit in c.get("citations", []):
            if cit.get("matched_scope") not in GRAPH_SCOPES:
                continue
            is_hang = cit.get("paragraph_type") == "항"
            para = cit.get("paragraph_num") if is_hang else None
            ho = cit.get("sub_ho") if is_hang else (cit.get("paragraph_num") if cit.get("paragraph_type") == "호" else None)
            graph[src].add((
                cit.get("ref_article_num"),
                cit.get("article_title"),
                cit.get("ref_section"),
                cit.get("ref_section_name"),
                para,
                ho,
            ))
    nz = lambda x: x if x is not None else -1
    return {
        f"{k[0]}|{k[1]}|{k[2]}|{k[3]}|{k[4]}|{k[5]}": sorted(v, key=lambda x: (nz(x[0]), nz(x[4]), nz(x[5])))
        for k, v in graph.items()
    }

graph_dog = build_reference_graph(chunks_dog)
graph_cat = build_reference_graph(chunks_cat)

print("강아지 그래프 노드 수:", len(graph_dog))
print("고양이 그래프 노드 수:", len(graph_cat))


강아지 그래프 노드 수: 446
고양이 그래프 노드 수: 408


In [3]:
# ===== 3: 조 제목 줄바꿈 정리 (튜플 6요소: num/title/section/section_name/항/호) =====

def clean_title(title):
    return title.replace("\n", " ").strip() if title else title

graph_dog_clean = {
    k: [[num, clean_title(title), sec, sname, para, ho] for num, title, sec, sname, para, ho in v]
    for k, v in graph_dog.items()
}
graph_cat_clean = {
    k: [[num, clean_title(title), sec, sname, para, ho] for num, title, sec, sname, para, ho in v]
    for k, v in graph_cat.items()
}
print("정리 완료 — 강아지 노드:", len(graph_dog_clean), "고양이 노드:", len(graph_cat_clean))


정리 완료 — 강아지 노드: 446 고양이 노드: 408


In [4]:
# ===== 4: 정리된 그래프 저장 =====

with open(GRAPH_DIR / "petmint_강아지_graph.json", "w", encoding="utf-8") as f:
    json.dump(graph_dog_clean, f, ensure_ascii=False, indent=2)

with open(GRAPH_DIR / "petmint_고양이_graph.json", "w", encoding="utf-8") as f:
    json.dump(graph_cat_clean, f, ensure_ascii=False, indent=2)

print("강아지 그래프 노드 수:", len(graph_dog_clean))
print("고양이 그래프 노드 수:", len(graph_cat_clean))
print("저장 완료:", GRAPH_DIR)

강아지 그래프 노드 수: 446
고양이 그래프 노드 수: 408
저장 완료: /home/kopo17/AI_programming/petmint-rag-chatbot/data/graph
